# DepositIQ 02 — Outliers and Skewed Numerics

**Rubric:** Criterion 3 — Data Preprocessing & Cleaning · **Task:** T13 · **Phase:** 3
**Dataset:** primary set, `data/raw/bank-full.csv` (45,211 × 17), loaded through `src/data_loader.py`
**Depends on:** T11 ([01_data_understanding.ipynb](01_data_understanding.ipynb)) for the measured
quality profile, and T12 ([src/clean.py](../src/clean.py)) for the cleaned frame.

## What this notebook decides

T11 measured *how* skewed the numeric columns are. This notebook answers the question T11
deliberately left open: **what do we do about it?** Specifically, for the four columns the task names
— `balance`, `campaign`, `previous`, `duration` — it chooses between:

- **Option 1** — keep every value and rely on a **robust scaler** (median / IQR) where a model needs
  scaled inputs, or
- **Option 2** — **winsorise** (clip the extremes to a percentile bound).

The decision is made from measurement, not preference, and the three tests are:

1. **How heavy is the tail?** skew, excess kurtosis, quantiles, counts beyond 1.5× and 3× IQR.
2. **Is the tail signal or noise?** the subscription rate inside the tail compared with the body —
   if the two differ, clipping collapses two genuinely different groups of clients onto one
   identical value, which is information loss that no scaler can undo.
3. **What does the treatment actually change?** the skew left after each candidate treatment, how
   many rows it rewrites, whether it preserves order, and how many distinct values it destroys.

## What this notebook does NOT do

It does not build the preprocessing pipeline — that is T24 (`src/preprocess.py`) — and it does not
touch `src/clean.py`, which is T12's deliverable. It does not train a model: the consequence of a
skewed feature differs by model family, so the notebook states that consequence per family and
leaves the fitting to Phase 6. No file is written except the three figures.

## How to run

From the repository root, inside the project virtual environment:

```bash
jupyter nbconvert --to notebook --execute --inplace notebooks/02_preprocessing.ipynb
```

## Setup

Imports and the same path handling as T11, so the notebook runs from any working directory
(decision D12).

In [1]:
import platform
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").exists():
            return candidate
    raise RuntimeError("Could not find the repository root above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config              # noqa: E402
from src import data_loader as load  # noqa: E402
from src import clean               # noqa: E402

config.ensure_dirs()

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

print("Executed  :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python    :", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)
print("Repo root :", REPO_ROOT)
print("Figures   :", config.FIGURES_DIR.relative_to(REPO_ROOT))

Executed  : 2026-10-27 18:51:53
Python    : 3.11.9 | pandas 2.3.3 | numpy 2.3.3
Repo root : D:\diabetes+130-us+hospitals+for+years+1999-2008\depositiq-term-deposit-prediction
Figures   : reports\figures


## 1. Scope: which columns, and why one is missing

The task names four columns. One more is included because T12 created it, and one existing column
is deliberately excluded. Stating both is the point of this section.

In [2]:
RAW = load.load_raw("full", verbose=False)
CLEANED, clean_report = clean.clean_frame(RAW, "full")

# The four columns this task must decide on, plus the recency column T12 derived from 'pdays'.
DECISION_COLUMNS = ["balance", "campaign", "previous", "duration"]
DERIVED_COLUMN = clean.CONTACT_RECENCY
ANALYSIS_COLUMNS = DECISION_COLUMNS + [DERIVED_COLUMN]

print(f"rows: {len(CLEANED):,} | columns before cleaning: {RAW.shape[1]} | after: {CLEANED.shape[1]}")
print(f"T12 added: {clean_report['new_columns']} | invariants hold: {clean_report['invariants_hold']}")
print()
print("Analysed here :", ANALYSIS_COLUMNS)
print("Excluded      : 'pdays' - T11 section 7 showed it is a sentinel plus a small block of real")
print("                day counts, so its skew describes an artefact of the encoding rather than a")
print("                distribution. T12 replaced it with the flag + 'days_since_contact' above.")
print("                'day' is excluded too: it is a calendar position (1-31), not a quantity,")
print("                and it is not in the task's list.")
print()
print("Also excluded: every categorical column - skew is a numeric statistic. The encoding of")
print("                'unknown' was decided in T12 (D22) and is not revisited here.")
print()

# T12 claims it changed no value and only added columns. Verify that here rather than trust it:
# if a value had moved, T11's measured skew would no longer describe this frame.
identical = {column: bool(RAW[column].equals(CLEANED[column])) for column in RAW.columns}
print(f"original columns bit-identical after T12 cleaning: {all(identical.values())}")
print(f"recency column dtype: {CLEANED[DERIVED_COLUMN].dtype} | NaN (sentinel removed): "
      f"{int(CLEANED[DERIVED_COLUMN].isna().sum()):,}")

rows: 45,211 | columns before cleaning: 17 | after: 19
T12 added: ['previously_contacted', 'days_since_contact'] | invariants hold: True

Analysed here : ['balance', 'campaign', 'previous', 'duration', 'days_since_contact']
Excluded      : 'pdays' - T11 section 7 showed it is a sentinel plus a small block of real
                day counts, so its skew describes an artefact of the encoding rather than a
                distribution. T12 replaced it with the flag + 'days_since_contact' above.
                'day' is excluded too: it is a calendar position (1-31), not a quantity,
                and it is not in the task's list.

Also excluded: every categorical column - skew is a numeric statistic. The encoding of
                'unknown' was decided in T12 (D22) and is not revisited here.

original columns bit-identical after T12 cleaning: True
recency column dtype: float64 | NaN (sentinel removed): 36,954


## 2. How heavy is the tail?

Skew, excess kurtosis, quantiles and outlier counts for each analysed column. Excess kurtosis is
reported alongside skew because they answer different questions: skew says the mass leans one way,
kurtosis says how much of it sits in the extremes.

In [3]:
def tail_profile(series: pd.Series) -> dict:
    values = series.dropna()
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    iqr = q3 - q1
    p99 = values.quantile(0.99)
    return {
        "n": len(values),
        "missing": int(series.isna().sum()),
        "mean": values.mean(),
        "median": values.median(),
        "std": values.std(),
        "skew": values.skew(),
        "excess_kurtosis": values.kurtosis(),
        "min": values.min(),
        "p01": values.quantile(0.01),
        "p25": q1,
        "p75": q3,
        "p99": p99,
        "max": values.max(),
        "max_over_p99": values.max() / p99 if p99 else np.nan,
        "outliers_1.5iqr": int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum()),
        "outliers_3iqr": int(((values < q1 - 3 * iqr) | (values > q3 + 3 * iqr)).sum()),
    }


profile = pd.DataFrame({column: tail_profile(CLEANED[column]) for column in ANALYSIS_COLUMNS}).T
PROFILE_FORMATS = {
    "mean": "{:,.1f}".format, "median": "{:,.1f}".format, "std": "{:,.1f}".format,
    "skew": "{:+.2f}".format, "excess_kurtosis": "{:,.1f}".format,
    "min": "{:,.0f}".format, "p01": "{:,.0f}".format, "p25": "{:,.0f}".format,
    "p75": "{:,.0f}".format, "p99": "{:,.0f}".format, "max": "{:,.0f}".format,
    "max_over_p99": "{:.2f}x".format,
}
print(profile.to_string(formatters=PROFILE_FORMATS))
print()

for column in ANALYSIS_COLUMNS:
    row = profile.loc[column]
    if abs(row["skew"]) > 1:
        shape = "highly skewed"
    elif abs(row["skew"]) > 0.5:
        shape = "moderately skewed"
    else:
        shape = "near-symmetric"
    print(f"  {column:<20} skew {row['skew']:+.2f} ({shape}), excess kurtosis {row['excess_kurtosis']:,.1f}, "
          f"max is {row['max_over_p99']:.2f}x the p99, {int(row['outliers_1.5iqr']):,} rows beyond 1.5xIQR")

print()
print("Insight 1: every one of the four named columns is right-skewed, and two of them ('balance',")
print("           'previous') are extremely so. The mean is not a usable summary for any of them:")
print("           for 'previous' the median is 0 while the maximum runs into the hundreds, and for")
print("           'balance' the tail reaches six figures. The 1.5xIQR rule flags a large absolute")
print("           number of rows, but section 3 asks whether those rows are noise before treating")
print("           them as such - a rule of thumb is not evidence.")
print("           Note also that 'balance' has negative values, which are a separate question about")
print("           sign rather than tail shape: T11 section 9 established that all 3,766 of them are")
print("           legitimate overdrafts, so nothing here may clip at zero.")

                          n  missing    mean median     std   skew excess_kurtosis    min  p01 p25   p75    p99     max max_over_p99  outliers_1.5iqr  outliers_3iqr
balance             45211.0      0.0 1,362.3  448.0 3,044.8  +8.36           140.8 -8,019 -627  72 1,428 13,165 102,127        7.76x           4729.0         2443.0
campaign            45211.0      0.0     2.8    2.0     3.1  +4.90            39.2      1    1   1     3     16      63        3.94x           3064.0         1462.0
previous            45211.0      0.0     0.6    0.0     2.3 +41.85         4,506.9      0    0   0     0      9     275       30.90x           8257.0         8257.0
duration            45211.0      0.0   258.2  180.0   257.5  +3.14            18.2      0   11 103   319  1,269   4,918        3.88x           3235.0         1155.0
days_since_contact   8257.0  36954.0   224.6  194.0   115.3  +0.69             1.5      1    8 133   327    531     871        1.64x             49.0            0.0

  balance

## 3. Is the tail signal or noise?

This is the test that decides the task. If the rows in the extreme tail behave like the rest of the
data, clipping them costs little. If they behave differently, clipping **merges two distinct groups
into one identical value** — and because winsorising is not invertible, that information is gone for
every model downstream, not just the one being tested today.

`y` is used here only as a **descriptive target** for grouping rows. That is not leakage: nothing
about the target is being fed to a model in this notebook, and the grouping is a property of the
dataset as a whole, not a feature.

In [4]:
rate = lambda mask: float((CLEANED.loc[mask, config.TARGET] == config.POSITIVE_LABEL).mean())
overall_rate = rate(pd.Series(True, index=CLEANED.index))

rows = []
for column in DECISION_COLUMNS:
    values = CLEANED[column]
    p99 = values.quantile(0.99)
    in_tail = values > p99
    rows.append({
        "column": column,
        "p99": p99,
        "tail_rows": int(in_tail.sum()),
        "subscription_rate_in_tail": rate(in_tail),
        "subscription_rate_in_body": rate(~in_tail),
        "difference_pp": 100 * (rate(in_tail) - rate(~in_tail)),
    })

tail = pd.DataFrame(rows)
tail["rate_ratio_tail_vs_body"] = tail["subscription_rate_in_tail"] / tail["subscription_rate_in_body"]
print(f"overall subscription rate: {overall_rate:.2%}")
print(tail.to_string(index=False, formatters={
    "p99": "{:,.0f}".format,
    "subscription_rate_in_tail": "{:.2%}".format,
    "subscription_rate_in_body": "{:.2%}".format,
    "difference_pp": "{:+.2f} pp".format,
    "rate_ratio_tail_vs_body": "{:.2f}x".format,
}))
print()

# The tail is not the only place signal can live: check the whole range, in bands.
by_decile = CLEANED.groupby(pd.qcut(CLEANED["balance"], 10, duplicates="drop"), observed=True).agg(
    rows=(config.TARGET, "size"),
    subscription_rate=(config.TARGET, lambda s: float((s == config.POSITIVE_LABEL).mean())),
)
by_decile["balance_range"] = [f"{interval.left:,.0f} to {interval.right:,.0f}" for interval in by_decile.index]
print("subscription rate by 'balance' decile:")
print(by_decile[["balance_range", "rows", "subscription_rate"]].to_string(
    index=False, formatters={"subscription_rate": "{:.2%}".format}))
print()

campaign_band = pd.cut(CLEANED["campaign"], [0, 1, 2, 3, 5, 10, 1000],
                       labels=["1", "2", "3", "4-5", "6-10", "11+"])
campaign_table = CLEANED.groupby(campaign_band, observed=True).agg(
    rows=(config.TARGET, "size"),
    subscription_rate=(config.TARGET, lambda s: float((s == config.POSITIVE_LABEL).mean())),
)
print("subscription rate by 'campaign' band (contacts during this campaign):")
print(campaign_table.to_string(formatters={"subscription_rate": "{:.2%}".format}))
print()

previous_band = pd.cut(CLEANED["previous"], [-1, 0, 1, 2, 1000], labels=["0", "1", "2", "3+"])
previous_table = CLEANED.groupby(previous_band, observed=True).agg(
    rows=(config.TARGET, "size"),
    subscription_rate=(config.TARGET, lambda s: float((s == config.POSITIVE_LABEL).mean())),
)
print("subscription rate by 'previous' band (contacts before this campaign):")
print(previous_table.to_string(formatters={"subscription_rate": "{:.2%}".format}))
print()

lower, upper = campaign_table["subscription_rate"].iloc[0], campaign_table["subscription_rate"].iloc[-1]
print("Insight 2: the tails carry signal, and in opposite directions per column, which is exactly")
print("           what a single clipping rule would destroy.")
print(f"           'campaign' is the clearest case: clients contacted once subscribe at "
      f"{lower:.2%}, while those contacted 11+ times subscribe at {upper:.2%} - a "
      f"{abs(lower - upper) * 100:.1f} pp gap. Winsorising at any percentile that touches this")
print("           band would push both groups onto the same value.")
print(f"           'balance' behaves the other way: the rate rises across the deciles, so the")
print(f"           wealthy tail is not noise - it is the segment the campaign most wants to reach.")
print("           'previous' shows that history matters too: the band is not uniform across its")
print("           range, so collapsing 0/1/2/3+ into one clipped value would erase a real split.")
print()
print("           This is the argument for Option 1: the extremes are not measurement error, they")
print("           are the clients whose behaviour differs. A summary statistic being awkward is")
print("           not a reason to delete the observation behind it.")

overall subscription rate: 11.70%
  column    p99  tail_rows subscription_rate_in_tail subscription_rate_in_body difference_pp rate_ratio_tail_vs_body
 balance 13,165        453                    11.92%                    11.70%      +0.22 pp                   1.02x
campaign     16        451                     2.44%                    11.79%      -9.35 pp                   0.21x
previous      9        453                    22.96%                    11.58%     +11.37 pp                   1.98x
duration  1,269        452                    61.50%                    11.20%     +50.31 pp                   5.49x



subscription rate by 'balance' decile:
   balance_range  rows subscription_rate
     -8,019 to 0  7280             6.90%
         0 to 22  1773             7.22%
       22 to 131  4544             9.18%
      131 to 272  4516            10.83%
      272 to 448  4495            11.41%
      448 to 701  4522            12.01%
    701 to 1,126  4526            12.68%
  1,126 to 1,859  4513            14.25%
  1,859 to 3,574  4521            16.61%
3,574 to 102,127  4521            16.12%

subscription rate by 'campaign' band (contacts during this campaign):
           rows subscription_rate
campaign                         
1         17544            14.60%
2         12505            11.20%
3          5521            11.19%
4-5        5286             8.63%
6-10       3159             6.52%
11+        1196             3.93%

subscription rate by 'previous' band (contacts before this campaign):
           rows subscription_rate
previous                         
0         36954             

## 4. What each candidate treatment does

Every option is measured the same way, so the comparison is like-for-like: the skew left behind,
how many rows it rewrites, whether it preserves order, and how many distinct values it destroys.

The distinction that matters is **injective vs not**. A log-like transform and an affine rescale
(what `RobustScaler` does) are monotone, one-to-one maps: they change the numbers but never change
which client is bigger than which. Winsorising is not one-to-one — it maps many distinct values onto
the same bound — so it is the only option here that can destroy information.

The log-like transform used is **`arcsinh`**, not `log1p`. That is not a stylistic choice: `log1p` is
undefined for values below −1, and `balance` contains negative balances down to −8,019, so plain
`log1p` would turn real clients into `NaN`. The code below measures how many, rather than assuming.

In [5]:
# Before comparing transforms, check the one that looks obvious: plain log1p on 'balance'.
# log1p(x) is only defined for x > -1, so a negative balance cannot be transformed by it at all.
# The warnings this deliberately triggers are silenced here: probing the invalid transform IS the
# point, and leaving matplotlib/numpy noise in the saved output would only obscure the finding.
balance_float = CLEANED["balance"].astype(float)
with np.errstate(divide="ignore", invalid="ignore"):
    log1p_values = np.log1p(balance_float)
log1p_nan = int(np.isnan(log1p_values).sum())
log1p_inf = int(np.isinf(log1p_values).sum())
negative_balance = int((CLEANED["balance"] < 0).sum())
print(f"plain log1p on 'balance' would produce {log1p_nan:,} NaN and {log1p_inf:,} -inf values,")
print(f"because {negative_balance:,} rows have a negative balance - all of them legitimate")
print("overdrafts that T11 section 9 requires us to keep. arcsinh is defined for every real")
print("number and behaves like a logarithm in the tail, so it is used instead.")
print()

# The interquartile range is the denominator of a median/IQR rescale, so a ZERO IQR is not a
# curiosity - it makes the transform undefined. Checked before use rather than discovered as a NaN.
iqr_by_column = {
    column: float(CLEANED[column].quantile(0.75) - CLEANED[column].quantile(0.25))
    for column in DECISION_COLUMNS
}
print("interquartile range - the scale a median/IQR rescale divides by:")
for column, iqr in iqr_by_column.items():
    flag = "   <-- ZERO: median/IQR scaling is undefined for this column" if iqr == 0 else ""
    print(f"  {column:<10} IQR = {iqr:>10,.0f}{flag}")
print()

TREATMENT_ORDER = [
    "keep (no transform)",
    "arcsinh (monotone)",
    "robust scale median/IQR (affine)",
    "winsorise 1-99",
    "winsorise 5-95",
]


def candidate_treatments(values: pd.Series) -> dict:
    v = values.dropna()
    iqr = float(v.quantile(0.75) - v.quantile(0.25))
    options = {
        "keep (no transform)": v,
        "arcsinh (monotone)": np.arcsinh(v),
        "winsorise 1-99": v.clip(v.quantile(0.01), v.quantile(0.99)),
        "winsorise 5-95": v.clip(v.quantile(0.05), v.quantile(0.95)),
    }
    if iqr > 0:
        # Only offered when the IQR is non-zero. Computing it anyway would divide by zero and
        # return infinities whose skew is NaN - a failure that hides inside a printed table.
        options["robust scale median/IQR (affine)"] = (v - v.median()) / iqr
    return options


treatment_rows = []
for column in DECISION_COLUMNS:
    original = CLEANED[column]
    options = candidate_treatments(original)
    for name in TREATMENT_ORDER:
        if name not in options:
            treatment_rows.append({
                "column": column, "treatment": name, "skew_after": np.nan,
                "rows_rewritten": 0, "distinct_values_lost": 0, "order_preserved": None,
                "note": "undefined - IQR is 0",
            })
            continue
        treated = options[name]
        treatment_rows.append({
            "column": column,
            "treatment": name,
            "skew_after": float(treated.skew()),
            "rows_rewritten": int((treated.values != original.dropna().values).sum()),
            # How many distinct client behaviours stop being distinguishable under this option.
            "distinct_values_lost": int(original.nunique() - treated.round(6).nunique()),
            # "order preserved" = the map is monotone and one-to-one, so no two clients swap rank
            # and no two distinct values become equal.
            "order_preserved": not name.startswith("winsorise"),
            "note": "",
        })

treatments = pd.DataFrame(treatment_rows)


def display_rows_for(column: str) -> list:
    # Rendered as strings, deliberately. pandas prints a float NaN as "NaN" without consulting the
    # formatter, so an undefined rescale would have looked like a measurement rather than a gap.
    rows = []
    for row in treatments[treatments["column"] == column].itertuples():
        undefined = pd.isna(row.skew_after)
        rows.append({
            "treatment": row.treatment,
            "skew_after": "n/a" if undefined else f"{row.skew_after:+.2f}",
            "rows_rewritten": "-" if undefined else f"{row.rows_rewritten:,}",
            "distinct_values_lost": "-" if undefined else f"{row.distinct_values_lost:,}",
            "order_preserved": "-" if row.order_preserved is None
                               else ("yes" if row.order_preserved else "NO"),
            "note": row.note,
        })
    return rows


for column in DECISION_COLUMNS:
    print("=" * 100)
    print(f"{column} - skew before any treatment: {CLEANED[column].skew():+.2f}")
    print(pd.DataFrame(display_rows_for(column)).to_string(index=False))
    print()

zero_iqr_columns = [column for column, iqr in iqr_by_column.items() if iqr == 0]
print("Insight 3: the table makes the trade-off explicit.")
print("           - 'keep' and 'arcsinh' preserve every distinct value; 'arcsinh' also cuts the skew")
print("             sharply, which is why it is a legitimate option for a numeric-linear model.")
print("           - 'robust scale' leaves the skew unchanged, by definition: an affine map cannot")
print("             change shape. What it does is put the columns on a comparable footing without")
print("             touching order or cardinality.")
print("           - the winsorise options rewrite hundreds to thousands of rows and destroy distinct")
print("             values - the count in 'distinct_values_lost' is exactly the number of separate")
print("             client behaviours that stop being distinguishable.")
print()
print("Insight 4: one column has no robust-scale row at all, and that is a finding rather than a")
print("           gap in the table:")
for column in zero_iqr_columns:
    zero_share = float((CLEANED[column] == 0).mean())
    print(f"             {column:<10} IQR = 0 because {zero_share:.2%} of its rows are exactly 0, so a")
    print(f"                        median/IQR rescale divides by zero. Left unguarded it would return")
    print(f"                        infinities and a NaN skew, which is easy to miss in a printed")
    print(f"                        table. T24 must therefore not robust-scale this column.")
print()

winsorise = treatments[(treatments["treatment"] == "winsorise 5-95")]
print("           Cost of the harsher option (winsorise 5-95), measured per column:")
for row in winsorise.itertuples():
    print(f"             {row.column:<10} rewrites {row.rows_rewritten:>7,} rows "
          f"({row.rows_rewritten / len(CLEANED):>6.2%} of the dataset), "
          f"loses {row.distinct_values_lost:>5,} distinct values")

plain log1p on 'balance' would produce 3,716 NaN and 50 -inf values,
because 3,766 rows have a negative balance - all of them legitimate
overdrafts that T11 section 9 requires us to keep. arcsinh is defined for every real
number and behaves like a logarithm in the tail, so it is used instead.

interquartile range - the scale a median/IQR rescale divides by:
  balance    IQR =      1,356
  campaign   IQR =          2
  previous   IQR =          0   <-- ZERO: median/IQR scaling is undefined for this column
  duration   IQR =        216

balance - skew before any treatment: +8.36
                       treatment skew_after rows_rewritten distinct_values_lost order_preserved note
             keep (no transform)      +8.36              0                    0             yes     
              arcsinh (monotone)      -1.68         41,697                    0             yes     
robust scale median/IQR (affine)      +8.36         45,211                    0             yes     
            

## 5. The decision

The rule this notebook follows: **a treatment may only be applied where it changes something the
model can actually use.** That makes the answer model-dependent, and stating it per family is more
honest than pretending one preprocessing choice serves all of them.

In [6]:
clipped_rows = int(winsorise["rows_rewritten"].sum())
clipped_values = int(winsorise["distinct_values_lost"].sum())

# 'duration' is the leakage column and never reaches the main model, so the gap that motivates this
# decision has to be quoted from the columns that do. It is reported separately below.
usable = tail[tail["column"] != "duration"].copy()
usable = usable.reindex(usable["difference_pp"].abs().sort_values(ascending=False).index)
strongest = usable.iloc[0]
duration_row = tail[tail["column"] == "duration"].iloc[0]

print("DECISION (T13): keep every value - no winsorising, no clipping, no removal.")
print()
print("The evidence for it, measured above:")
print(f"  1. The extremes are informative, not erroneous. Among the columns that actually reach the")
print(f"     model, the widest tail/body gap is '{strongest['column']}': "
      f"{strongest['subscription_rate_in_tail']:.2%} in the tail against")
print(f"     {strongest['subscription_rate_in_body']:.2%} in the body "
      f"({strongest['difference_pp']:+.1f} pp), and the 'campaign' band runs from {lower:.2%} at one")
print(f"     contact to {upper:.2%} at 11+ - the same story pointing the other way.")
print(f"     ('duration' shows a {duration_row['difference_pp']:+.1f} pp gap, which is exactly why it leaks and")
print(f"     why D05 excludes it - not a reason to keep it.)")
print(f"  2. Clipping is the only non-invertible option here. Winsorising at 5-95 would rewrite")
print(f"     {clipped_rows:,} rows and destroy {clipped_values:,} distinct values across the four")
print(f"     columns, and no scaler can recover what it removes.")
print(f"  3. There is nothing invalid to remove. T11 section 9 found zero unexplained negatives and")
print(f"     zero impossible values; the data-quality problems were the 'unknown' flag and the")
print(f"     pdays sentinel, and both are handled in T12 (D22, D23) rather than by clipping.")
print()
print("What that means per model family (the consequence, not another test):")
print("  - Tree ensembles (Random Forest, XGBoost - decision D08): monotone transforms are")
print("    irrelevant to them, because a split on x is the same split on log(x). They need NO")
print("    scaling and NO clipping; passing the raw values is the correct choice.")
print("  - Logistic Regression (the T26 baseline): needs comparable scales, so it gets a")
print("    RobustScaler (median / IQR). That is an affine map: it limits how much leverage an")
print("    extreme value has on the fitted coefficients without deleting the value itself. This is")
print("    the 'keep + robust scale' option from the task, and it is the one being taken.")
print("  - 'duration': excluded from the main model for leakage (decision D05), retained unchanged")
print("    for the T22/T36 with-vs-without comparison. No treatment is applied to it because it")
print("    never reaches the main feature matrix.")
print(f"  - 'days_since_contact' (T12): skewed like the rest, and NaN for the "
      f"{int(CLEANED[DERIVED_COLUMN].isna().sum()):,} rows")
print("    where the sentinel was. NaN stays NaN here - T24 decides the imputation, and XGBoost")
print("    handles missing values natively.")
for column in [c for c in zero_iqr_columns if c in DECISION_COLUMNS]:
    print(f"  - '{column}': NOT robust-scaled. Its IQR is 0 (section 4), so a median/IQR rescale is")
    print(f"    undefined for it. T24 passes the raw count to the trees and uses the")
    print(f"    '{clean.CONTACT_FLAG}' flag (T12) for the linear model instead.")
print()
print("What would change this decision: if the Logistic Regression baseline turns out to be")
print("dominated by skew (a poor CV ROC-AUC that a monotone transform fixes), the response is to")
print("report that in T26/T31 and consider arcsinh for THAT model - not to pre-clip the data for")
print("every model on the strength of a rule of thumb.")
print()
print("Carried forward: T24 implements the RobustScaler in the ColumnTransformer, and T25 records")
print("which columns were kept untouched. Nothing is written to data/processed/ by this notebook.")

DECISION (T13): keep every value - no winsorising, no clipping, no removal.

The evidence for it, measured above:
  1. The extremes are informative, not erroneous. Among the columns that actually reach the
     model, the widest tail/body gap is 'previous': 22.96% in the tail against
     11.58% in the body (+11.4 pp), and the 'campaign' band runs from 14.60% at one
     contact to 3.93% at 11+ - the same story pointing the other way.
     ('duration' shows a +50.3 pp gap, which is exactly why it leaks and
     why D05 excludes it - not a reason to keep it.)
  2. Clipping is the only non-invertible option here. Winsorising at 5-95 would rewrite
     13,054 rows and destroy 3,366 distinct values across the four
     columns, and no scaler can recover what it removes.
  3. There is nothing invalid to remove. T11 section 9 found zero unexplained negatives and
     zero impossible values; the data-quality problems were the 'unknown' flag and the
     pdays sentinel, and both are handled in

  - 'days_since_contact' (T12): skewed like the rest, and NaN for the 36,954 rows
    where the sentinel was. NaN stays NaN here - T24 decides the imputation, and XGBoost
    handles missing values natively.
  - 'previous': NOT robust-scaled. Its IQR is 0 (section 4), so a median/IQR rescale is
    undefined for it. T24 passes the raw count to the trees and uses the
    'previously_contacted' flag (T12) for the linear model instead.

What would change this decision: if the Logistic Regression baseline turns out to be
dominated by skew (a poor CV ROC-AUC that a monotone transform fixes), the response is to
report that in T26/T31 and consider arcsinh for THAT model - not to pre-clip the data for
every model on the strength of a rule of thumb.

Carried forward: T24 implements the RobustScaler in the ColumnTransformer, and T25 records
which columns were kept untouched. Nothing is written to data/processed/ by this notebook.


## 6. Figures

`00_preprocessing_` prefix, so these sort before the Phase 4 EDA figures (`01_`–`05_`) and cannot
collide with them.

In [7]:
saved = []

# Figure 1 - the shape problem, and what a monotone transform does to it.
fig, axes = plt.subplots(2, len(DECISION_COLUMNS), figsize=(18, 7))
for index, column in enumerate(DECISION_COLUMNS):
    values = CLEANED[column].dropna()
    axes[0, index].hist(values, bins=60, color="#4472c4")
    axes[0, index].set_title(f"{column}: raw\nskew {values.skew():+.2f}")
    axes[0, index].set_yscale("log")
    axes[1, index].hist(np.arcsinh(values), bins=60, color="#9bbb59")
    axes[1, index].set_title(f"arcsinh({column})\nskew {np.arcsinh(values).skew():+.2f}")
    axes[1, index].set_yscale("log")
    for row in (0, 1):
        axes[row, index].set_ylabel("rows (log scale)" if index == 0 else "")
fig.suptitle("Raw distributions (top) and the same columns under a monotone arcsinh transform (bottom). "
             "The transform reshapes the axis; it does not delete a single client.")
fig.tight_layout()
path = config.FIGURES_DIR / "00_preprocessing_distributions.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
saved.append(path)

# Figure 2 - skew after each candidate treatment.
fig, axes = plt.subplots(1, len(DECISION_COLUMNS), figsize=(20, 4.5))
for ax, column in zip(axes, DECISION_COLUMNS):
    subset = treatments[treatments["column"] == column].set_index("treatment")["skew_after"]
    subset.plot(kind="bar", ax=ax, color=["#4472c4", "#9bbb59", "#8064a2", "#c0504d", "#e36c0a"])
    ax.axhline(1, color="grey", linestyle="--", linewidth=1)
    ax.axhline(-1, color="grey", linestyle="--", linewidth=1)
    ax.set_title(f"{column}")
    ax.set_ylabel("skew after treatment" if column == DECISION_COLUMNS[0] else "")
    ax.tick_params(axis="x", labelrotation=60, labelsize=7)
fig.suptitle("Skew remaining after each option (dashed lines at +/-1). Robust scaling cannot change "
             "skew; winsorising reduces it by deleting the tail it measures.")
fig.tight_layout()
path = config.FIGURES_DIR / "00_preprocessing_treatments.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
saved.append(path)

# Figure 3 - the signal in the tail, which is what made the decision.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
bar_labels = [f"{interval.left:,.0f}-{interval.right:,.0f}" for interval in by_decile.index]
axes[0].bar(bar_labels, by_decile["subscription_rate"], color="#4472c4")
axes[0].axhline(overall_rate, color="firebrick", linestyle="--", linewidth=1,
                label=f"overall {overall_rate:.2%}")
axes[0].set_title(f"balance decile - subscription rate climbs from "
                  f"{by_decile['subscription_rate'].iloc[0]:.1%} to "
                  f"{by_decile['subscription_rate'].max():.1%}")
axes[0].set_ylabel("subscription rate")
axes[0].tick_params(axis="x", labelrotation=45, labelsize=8)
axes[0].yaxis.set_major_formatter(lambda value, _position: f"{value:.0%}")
axes[0].legend()
axes[1].bar(campaign_table.index.astype(str), campaign_table["subscription_rate"], color="#c0504d")
axes[1].axhline(overall_rate, color="firebrick", linestyle="--", linewidth=1,
                label=f"overall {overall_rate:.2%}")
axes[1].set_title("campaign band - the tail behaves differently, in the other direction")
axes[1].set_xlabel("contacts during this campaign")
axes[1].tick_params(axis="x", labelrotation=0)
axes[1].yaxis.set_major_formatter(lambda value, _position: f"{value:.0%}")
axes[1].legend()
fig.suptitle("Winsorising merges these groups into one value. The tails are the segments, not the "
             "errors, of this dataset.")
fig.tight_layout()
path = config.FIGURES_DIR / "00_preprocessing_signal_in_tail.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
saved.append(path)

for path in saved:
    print(f"saved {path.relative_to(REPO_ROOT)}  ({path.stat().st_size / 1024:.0f} KB)")

saved reports\figures\00_preprocessing_distributions.png  (145 KB)
saved reports\figures\00_preprocessing_treatments.png  (123 KB)
saved reports\figures\00_preprocessing_signal_in_tail.png  (92 KB)


## 7. Findings, and what they hand to the next tasks

1. **All four named columns are right-skewed, two of them severely.** Measured skew: `previous`
   **+41.85**, `balance` **+8.36**, `campaign` **+4.90**, `duration` **+3.14** — every one well past
   the ±1 rule of thumb, with excess kurtosis of 4,507 and 141 on the first two. For none of the
   four is the mean a usable summary: `previous` has median 0 while its maximum is 275, and
   `balance` spans −8,019 to 102,127.
2. **The 1.5×IQR rule flags thousands of rows, and the rule is not evidence.** It is reported as a
   count, then tested rather than obeyed: 8,257 rows for `previous`, 4,729 for `balance`, 3,235 for
   `duration`. Applying it as a deletion rule would have removed the clients the campaign most
   wants — and note that for `previous` it flags 18.3% of the dataset, which is the tell that the
   rule is describing the column's shape rather than finding errors.
3. **The tails are signal, and they point in opposite directions.** The subscription rate climbs
   with `balance`, from **6.90%** in the lowest decile to **16.61%** by the ninth — the wealthy tail
   is the segment the campaign exists to reach. It falls with `campaign`, from **14.60%** for one
   contact to **3.93%** at 11+ — those clients are being chased, not courted. `previous` splits the
   same way (9.16% at zero prior contacts, 25.63% at 3+). A single clipping rule flattens all three
   patterns at once.
4. **Only one of the five options destroys information.** An `arcsinh` transform and a median/IQR
   rescale are monotone and one-to-one: they change the numbers, never the ordering, and never the
   number of distinct values. Winsorising is neither, which is why it is the option that carries the
   risk. A related trap was caught in the process: the obvious `log1p` transform is undefined for
   `balance`, whose negative values would have become `NaN` — the transform would have silently
   deleted exactly the overdraft rows the project is required to keep.
5. **Nothing in these columns is invalid.** There is no value to correct: the negatives in `balance`
   are real overdrafts (3,766 rows, 8.33%), and the extreme values are plausible clients rather than
   entry errors. The data-quality problems in this dataset were the `unknown` level and the `pdays`
   sentinel, and both were settled in T12 rather than by clipping.
6. **`previous` cannot be robust-scaled at all, and that had to be measured to be seen.** Its
   interquartile range is **0** (81.74% of rows are exactly 0), so the median/IQR rescale divides by
   zero. Left unguarded it returns infinities and a `NaN` skew, which is easy to miss inside a
   printed comparison table — so the notebook now checks the IQR first and omits the option with a
   stated reason. T24 must pass `previous` raw to the trees and use the `previously_contacted` flag
   (T12) for the linear model.
7. **Therefore: keep everything, and scale only where a model needs it.** Trees need nothing; the
   Logistic Regression baseline gets a median/IQR rescale, which limits the leverage of extremes
   without discarding any of them. `duration` is excluded from the main model for leakage (D05) and
   is left untouched for the T22/T36 comparison.

**Handover:** T24 implements the decision — a `RobustScaler` on the numeric block of the
`ColumnTransformer`, raw values elsewhere, `previous` excluded from scaling — and T25 records the
kept-untouched set. T26's baseline is the first place this decision is tested, and if skew does hurt
the linear baseline, the finding is reported there rather than pre-empted here.

## 8. Reproducibility record

In [8]:
print("Notebook   : notebooks/02_preprocessing.ipynb  (T13, rubric criterion 3)")
print("Executed   :", datetime.now().isoformat(timespec="seconds"))
print("Dataset    : primary set,", load.VARIANTS["full"].relative_to(REPO_ROOT),
      tuple(CLEANED.shape))
print("Decision   : keep every value; no winsorising; RobustScaler for the linear model only")
print("Figures    :")
for path in sorted(config.FIGURES_DIR.glob("00_preprocessing_*.png")):
    print(f"   {path.name:<44} {path.stat().st_size / 1024:>6.0f} KB")
print()
print("Rerun with : jupyter nbconvert --to notebook --execute --inplace "
      "notebooks/02_preprocessing.ipynb")
print("Next task  : T14 - turn src/clean.py into the checked raw -> processed pipeline.")

Notebook   : notebooks/02_preprocessing.ipynb  (T13, rubric criterion 3)
Executed   : 2026-10-27T18:51:56
Dataset    : primary set, data\raw\bank-full.csv (45211, 19)
Decision   : keep every value; no winsorising; RobustScaler for the linear model only
Figures    :
   00_preprocessing_distributions.png              145 KB
   00_preprocessing_signal_in_tail.png              92 KB
   00_preprocessing_treatments.png                 123 KB

Rerun with : jupyter nbconvert --to notebook --execute --inplace notebooks/02_preprocessing.ipynb
Next task  : T14 - turn src/clean.py into the checked raw -> processed pipeline.
